In [12]:
import requests
from bs4 import BeautifulSoup

### Grabbing

In [13]:
url = "https://silenthill.fandom.com/wiki/Inspirational_works_of_Silent_Hill"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

print("Yoinking page...")
response = requests.get(url, headers=headers)

if response.status_code != 200:
    print(f"AHHHHHH status code: {response.status_code}")
    exit()

print(f"Page is responding: {response.status_code}\nVery nice.\n")

Yoinking page...
Page is responding: 200
Very nice.



### Parsing

In [14]:
# create variable where parsed response will live
tree = BeautifulSoup(response.text, "html.parser")

content = tree.find("div", class_="mw-parser-output")

In [15]:
results = []
current_header = None
current_items = []

### Walking

In [16]:
for element in content.children:
    #skipping plain text nodes
    if not hasattr(element, "name"):
        continue

    tag = element.name

    # H2 headers
    if tag == "h2":
        if current_header and current_items:
            results.append({
                "header": current_header,
                "level": "h2",
                "items": current_items
            })
        
        # grab text (stripped of html)
        current_header = element.get_text(strip=True)
        current_items = []

    # H3 headers
    elif tag == "h3":
        if current_header and current_items:
            results.append({
                "header": current_header,
                "level": "h3",
                "items": current_items
            })
        current_header = element.get_text(strip=True)
        current_items = []

    # List items
    elif tag in ("ul", "ol"):
        for li in element.find_all("li"):
            text = li.get_text(strip=True)
            # drop citations
            import re
            text = re.sub(r'\[\d+\]', '', text).strip()
            if text:
                current_items.append(text)
 
# save last section after loop
if current_header and current_items:
    results.append({
        "header": current_header,
        "level": "h3" if current_header else "h2",
        "items": current_items
    })

### Display

In [43]:

with open("../python/output.txt", "w") as f:
    for section in results:
        level_marker = "-" if section["level"] == "h2" else ">"
        f.write(f"\n\n{level_marker}  {section['header'].upper()}")
        for item in section["items"]:
                display = item
                f.write(f"\n- {display}")


In [44]:
# Need to fix list item with all the shtuff showing
print("=" * 70)
print("  INSPIRATIONAL WORKS OF SILENT HILL — SCRAPED DATA")
print("=" * 70)
 
for section in results:
    level_marker = "▶▶" if section["level"] == "h2" else "  ▷"
    print(f"\n{level_marker}  {section['header'].upper()}")
    print("  " + "─" * 60)
    for item in section["items"]:
        # Truncate very long items for readable terminal output
        display = item
        print(f"    • {display}")
    
 
print("\n" + "=" * 70)
print(f"  Total sections scraped: {len(results)}")
print(f"  Total list items scraped: {sum(len(s['items']) for s in results)}")
print("=" * 70)

  INSPIRATIONAL WORKS OF SILENT HILL — SCRAPED DATA

  ▷  FICTION/NON-FICTION[]
  ────────────────────────────────────────────────────────────
    • The Box Man(1973) byKōbō Abe: Inspired the dangling legs symbolic ofCheryl MasonandAlessa GillespieinSilent Hill 3's chapel; confirmed by artistMasahiro Ito.
    • Works byMasako Bandō: Inspired the oppressive atmosphere of the games.Shikoku(1996)Inugami(1996)
    • Shikoku(1996)
    • Inugami(1996)
    • Works byRobert Bloch:Psycho(1959)Strange Eons(1978)
    • Psycho(1959)
    • Strange Eons(1978)
    • Works byRay Bradbury:The Martian Chronicles(1950)Fahrenheit 451(1953)Something Wicked This Way Comes(1962): Possibly referenced with the "GMR"shirtinSilent Hill 3.
    • The Martian Chronicles(1950)
    • Fahrenheit 451(1953)
    • Something Wicked This Way Comes(1962): Possibly referenced with the "GMR"shirtinSilent Hill 3.
    • Works byJonathan Carroll:The Land of Laughs(1980)After Silence(1993)
    • The Land of Laughs(1980)
    • Aft